In [ ]:
import featuretools as ft
import pandas as pd
import numpy as np
import psutil

print(f"Beschikbare cores: {psutil.cpu_count()}")

trans = pd.read_csv("../data/transactions_2016_2017.csv", low_memory=False)
trans["order_date"] = pd.to_datetime(trans["order_date"])

# Beperk tot snelle primitives — geen depth=2
es = ft.EntitySet(id="clv")

es = es.add_dataframe(
    dataframe_name="transactions",
    dataframe=trans,
    index="sale_id",
    time_index="order_date",
    logical_types={
        "cust_id": "Categorical",
        "prod_brand": "Categorical",
        "prod_type_3": "Categorical",
    }
)

es = es.add_dataframe(
    dataframe_name="customers",
    dataframe=pd.DataFrame({"cust_id": trans["cust_id"].unique()}),
    index="cust_id"
)

es = es.add_relationship("customers", "cust_id", "transactions", "cust_id")

# Alleen snelle aggregaties — geen transform primitives
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="customers",
    agg_primitives=["sum", "mean", "max", "min", "std", "count", "num_unique"],
    trans_primitives=[],  # geen transform — veel trager
    max_depth=1,
    verbose=True,
    n_jobs=-1
)

print(f"Features gegenereerd: {feature_matrix.shape[1]}")
print(feature_matrix.head())

# Vergelijk met wat we al hebben
our_features = [
    "orders_per_day", "n_products", "avg_size", "total_revenue", ...
]
new_cols = [c for c in feature_matrix.columns if not any(f in c.lower() for f in our_features)]
print(f"\nMogelijk nieuwe features: {len(new_cols)}")
for c in new_cols[:20]:
    print(f"  {c}")

feature_matrix.to_csv("../data/featuretools_features.csv")


/tmp/ipykernel_33803/3674653634.py:6: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("../data/transactions_2016_2017.csv")
/workspaces/AdvancedAnalyticsTabular/.venv/lib/python3.12/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/AdvancedAnalyticsTabular/.venv/lib/python3.12/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/AdvancedAnalyticsTabular/.venv/lib/python3.12/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed ind

WoodworkNotInitError: Woodwork not initialized for this DataFrame. Initialize by calling DataFrame.ww.init